# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [2]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['HJLPXSXJHP', 'QBNJXVGZOC'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 8, 10, 12, 16, 24, 19, 24, 10,  8, 16],
       [17,  2, 14, 10, 24, 22,  7, 26, 15,  3]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 16,  8, 10, 24, 19, 24, 16, 12, 10],
       [ 0,  3, 15, 26,  7, 22, 24, 10, 14,  2]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[16,  8, 10, 24, 19, 24, 16, 12, 10,  8],
       [ 3, 15, 26,  7, 22, 24, 10, 14,  2, 17]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [3]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        '''
        todo
        
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        # ── Step 1: Embedding ──────────────────────────────
        enc_emb = self.embed_layer(enc_ids)   # (batch, enc_len, 64)
        dec_emb = self.embed_layer(dec_ids)   # (batch, dec_len, 64)

        # ── Step 2: Encoder ────────────────────────────────
        # 注意：return_sequences=True，所以 enc_out 保存了「每一步」的隐藏状态
        # 这是 Attention 的关键！普通 Seq2Seq 只用 enc_state（最后一步）
        enc_out, enc_state = self.encoder(enc_emb)
        # enc_out:   (batch, enc_len, 128)  ← 所有时间步的 h
        # enc_state: (batch, 128)           ← 最后一步的 h

        # ── Step 3: Decoder ────────────────────────────────
        dec_out, dec_state = self.decoder(dec_emb, initial_state=[enc_state])
        # dec_out: (batch, dec_len, 128)

        # ── Step 4: 计算 Attention ──────────────────────────
        # 双线性 Attention：score(dec_h, enc_h) = dec_h @ W @ enc_h^T
        # 先用 dense_attn 对 enc_out 做线性变换（相当于乘以 W）
        enc_out_w = self.dense_attn(enc_out)  # (batch, enc_len, 128)

        # 矩阵乘法：(batch, dec_len, 128) x (batch, 128, enc_len)
        #         → (batch, dec_len, enc_len)
        # 含义：dec_out 的每个时间步 对 enc_out 的每个时间步 的「注意力分数」
        attn_scores = tf.matmul(dec_out, enc_out_w, transpose_b=True)

        # Softmax：让每行（每个 decoder 步骤对所有 encoder 步骤）的权重加起来=1
        attn_weights = tf.nn.softmax(attn_scores, axis=-1)
        # attn_weights: (batch, dec_len, enc_len)

        # 加权求和：用注意力权重对 enc_out 做加权平均 → context vector
        # (batch, dec_len, enc_len) x (batch, enc_len, 128) → (batch, dec_len, 128)
        context = tf.matmul(attn_weights, enc_out)

        # ── Step 5: 融合 decoder 输出 和 context vector ──────
        combined = dec_out + context          # (batch, dec_len, 128)

        # ── Step 6: 输出层 ──────────────────────────────────
        logits = self.dense(combined)         # (batch, dec_len, 27)
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        '''
    
        '''
        todo
        参考sequence_reversal-exercise, 自己构建单步解码逻辑'''
        # ── Step 1: Embedding + RNN 一步 ─────────────────────
        inp_emb = self.embed_layer(x)                       # (batch, 64)
        h, state = self.decoder_cell.call(inp_emb, state)   # h: (batch, 128)

        # ── Step 2: 计算 Attention ──────────────────────────
        enc_out_w = self.dense_attn(enc_out)                # (batch, enc_len, 128)

        # 扩展 h 为 (batch, 1, 128)，方便矩阵乘法
        h_exp = tf.expand_dims(h, axis=1)                   # (batch, 1, 128)

        # score: (batch, 1, enc_len)
        attn_scores = tf.matmul(h_exp, enc_out_w, transpose_b=True)
        attn_weights = tf.nn.softmax(attn_scores, axis=-1)  # (batch, 1, enc_len)

        # context: (batch, 1, 128) → squeeze → (batch, 128)
        context = tf.squeeze(
            tf.matmul(attn_weights, enc_out), axis=1
        )

        # ── Step 3: 融合并输出 ──────────────────────────────
        combined = h + context                              # (batch, 128)
        logits = self.dense(combined)                       # (batch, 27)
        out = tf.argmax(logits, axis=-1)                    # (batch,)
        return out, state

# Loss函数以及训练逻辑

In [4]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [5]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3061295
step 500 : loss 1.551889
step 1000 : loss 0.42651433
step 1500 : loss 0.080680326


<tf.Tensor: shape=(), dtype=float32, numpy=0.032654632>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [6]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, False, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('GCUSOXYORXRSKFCUXBFA', 'AFBXUCFKSRXROYXOSUCG'), ('EMLHCHFFJHOAGENCGVOX', 'XOVGCNEGAOHJFFHCHLME'), ('OFKJABGDSUELOFSMCWKO', 'OKWCMSFOLEUSDGBAJKFO'), ('GXNWJKKAKDMSXBIIMZPM', 'MPZMIIBXSMDKAKKJWNXG'), ('LVMUGQQZBCQDGYDAFIHO', 'OHIFADYGDQCBZQQGUMVL'), ('BRQGHJOGHJDPKQSROQMO', 'OMQORSQKPDJHGOJHGQRB'), ('AZMPEWNAJBHFEGABOMUS', 'SUMOBAGEFHBJANWEPMZA'), ('GMYFUXXLOQBJYGCWHLWP', 'PWLHWCGYJBQOLXXUFYMG'), ('KMZKSSAHKAHJHTHVKIYS', 'SYIKVHTHJHAKHASSKZMK'), ('RNSJCVVCWMWANYEAVTGW', 'WGTVAEYNAWMWCVVCJSNR'), ('VWGHDGKCVBBUGLOVVDCK', 'KCDVVOLGUBBVCKGDHGWV'), ('SRRQZRIQSTYKUUTTPHJD', 'DKHPTTUUKYTSQIRZQRRS'), ('UEIJIFIIALKPUZVIVWUJ', 'JUWVIVZUPKLAIIFIJIEU'), ('DSEUUBOOULZLFYKDWZYQ', 'QYZWDKYFLZLUOOBUUESD'), ('EDITCNSLNMSHBXYDNQLJ', 'JLQNDYXBHSMNLSNCTIDE'), ('MWXLSDKFKMAMDQFNOTBE', 'EBTONFQDMAMKFKDSLXWM'), ('RLO